[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/fresnel/fresnel.ipynb)

# Fresnel Wave Surfaces

This example computes the Fresnel wave surfaces of isotropic, uniaxial, and biaxial crystals. Monochromatic light propagating in one direction through an anisotropic dielectric generally supports two wavelengths corresponding to two polarizations, and sweeping over all propagation directions traces the two sheets of the wave surface. Both sheets coincide in vacuum, touch along a single optic axis in a uniaxial crystal, and intersect along two optic axes in a biaxial crystal.

The electromagnetic field of a plane wave is modelled in spacetime as the bivector `k ^ A` of a wavevector `k` and potential `A`, while a linear medium maps fields to excitation antibivectors through area-patch dyads `patch * (Bivector & patch)` weighted by principal permittivities and reluctivity. Imposing the source-free Maxwell equation forms the potential map `k ^ medium(k ^ A)` from extrinsic potentials to intrinsic 4-currents, whose kernel always contains gauge potentials parallel to `k`. A propagating wave adds a second null direction so the induced 3-volume outermorphism `wave.outermorphism(Trivector)` vanishes, and pairing that map on the observer's rest space divides out `(k & time).squared()` to yield the metric-free Fresnel quartic ([Itin, 2009](https://arxiv.org/abs/0908.0922)).

In the tensor index notation of premetric electrodynamics, substituting $F_{cd} = k_c A_d - k_d A_c$ into the excitation tensor density $\mathcal{H}^{ab} = \tfrac{1}{2}\chi^{abcd}F_{cd}$ gives the wave operator $M^{ab}(k) = \chi^{acbd}k_c k_d$ on $A_b$, whose cofactor matrix requires double contraction with the Levi-Civita symbols $\operatorname{adj}(M)_{ab} = \tfrac{1}{6}\epsilon_{ac_1 c_2 c_3}\epsilon_{bd_1 d_2 d_3} M^{c_1 d_1} M^{c_2 d_2} M^{c_3 d_3} = \mathcal{G}^{cdef} k_c k_d k_e k_f\, k_a k_b$ to extract the Tamm–Rubilar dispersion scalar $\mathcal{G}^{cdef} k_c k_d k_e k_f = 0$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga

from examples.electromagnetism.fresnel import render
from examples.util.animation import save_figure

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                     # Scalar+
Vector = ga.gatype.vector()                                     # Vector+
Bivector = ga.gatype.bivector()                                 # Bivector+
Antibivector = ga.gatype.antibivector()                         # Antibivector-
Trivector = ga.gatype.trivector()                               # Trivector+
# Potentials are extrinsic vectors; the 4-currents they drive are intrinsic antivectors.
Antivector = ga.gatype.antivector()                             # Antivector-
Constitutive = ga.gatype((Antibivector, Bivector))              # Antibivector- <- Bivector+
WaveMap = ga.gatype((Antivector, Vector))                       # Antivector- <- Vector+
# A chiral coupling of excitation to field carries intrinsic scalar orientation.
AxionCoupling = ga.gatype.antipseudoscalar()                    # Scalar-

# Unit timelike direction, spatial axis directions, and spatial coordinate planes of the rest observer.
time = -mv.t.dual()                                             # [] Antivector-
axes = stack((mv.x.dual(), mv.y.dual(), mv.z.dual()))           # [3] Antivector-
magnetic_planes = stack((mv.yz, mv.zx, mv.xy))                  # [3] Bivector+
# Intrinsic worldsheet patches read electric components; intrinsic spatial patches read magnetic components.
swept_planes = axes & time                                      # [3] Antibivector-
patches = magnetic_planes.dual()                                # [3] Antibivector-
rest_space = mv.xyz                                             # [] Trivector+
root_signs = np.array([-1, 1])

## Constitutive maps of rest-frame dielectrics

A rest-frame dielectric maps each spacetime field bivector to an excitation antibivector through a sum of six area-patch dyads `patch * (Bivector & patch)`. The three worldsheet patches `axes & time` swept by the spatial axes through time read the electric field components and carry the principal permittivities, while the three spatial plane patches read the magnetic field components and share the scalar reluctivity.

In [ ]:
# Principal permittivities along x, y, and z and scalar reluctivities for vacuum, uniaxial, and biaxial media.
permittivities = np.array([[1, 1, 1],
                           [2.25, 2.25, 3.24],
                           [1.44, 2.25, 3.24]])
reluctivities = np.array([1, 1, 1])
n_materials = len(permittivities)

In [ ]:
def dielectric(permittivity: np.ndarray, reluctivity: np.ndarray) -> Constitutive:
    """Electric swept-plane dyads weighted by `permittivity` (`[..., 3]`) and magnetic spatial-patch dyads weighted by `reluctivity` (`[...]`) sum to the rest-frame constitutive map (`[...] Antibivector- <- Bivector+`)."""
    # Time squares to -1, so electric swept-plane dyads enter with a minus sign.
    electric = -(permittivity * swept_planes * (Bivector & swept_planes)).sum(axis=-1)  # [...] Antibivector- <- Bivector+
    magnetic = (patches * (Bivector & patches)).sum(axis=-1)                            # [] Antibivector- <- Bivector+
    return electric + reluctivity * magnetic                                            # [...] Antibivector- <- Bivector+


media = dielectric(permittivities, reluctivities)                               # [n_materials] Antibivector- <- Bivector+

## The potential map and the Fresnel quartic

For a wavevector `k`, wedging with a potential `A` gives the field `k ^ A`, and the source-free Maxwell equation requires the driven 4-current `k ^ medium(k ^ A)` to vanish. Because any gauge potential parallel to `k` already gives zero field, this potential map `Antivector <- Vector` always has at least a one-dimensional kernel. A physical wave exists when a second independent potential lies in the kernel, which happens exactly when the induced 3-volume outermorphism `wave.outermorphism(Trivector)` sends every potential 3-volume to zero. Pairing that volume map on `rest_space` divides out `(k & time).squared()` and isolates the scalar Fresnel quartic.

In [ ]:
def potential_map(medium: Constitutive, wavevector: Vector) -> WaveMap:
    """Wedging a potential with `wavevector` and applying `medium` maps each candidate potential to the 4-current it would require (`[...] Antivector- <- Vector+`)."""
    field = wavevector ^ Vector                                               # [...] Bivector+ <- Vector+
    return wavevector ^ medium(field)                                         # [...] Antivector- <- Vector+


def fresnel_quartic(
    medium: Constitutive,
    observer_time: Antivector,
    observer_space: Trivector,
    wavevector: Vector,
) -> Scalar:
    """Volume action of the potential map on `observer_space`, normalized by squared observer frequency, gives the Fresnel quartic (`[...] Scalar+`)."""
    wave = potential_map(medium, wavevector)                                  # [...] Antivector- <- Vector+
    # A physical wave adds a second null direction beyond the gauge wavevector, so 3-volumes collapse.
    volumes = wave.outermorphism(Trivector)                                   # [...] Vector- <- Trivector+
    measured_frequency = wavevector & observer_time                           # [...] Scalar+
    return (observer_space & volumes(observer_space)) / measured_frequency.squared()  # [...] Scalar+

## Symmetry, adjugate, and adjoint in spacetime

Pairing one potential against the 4-current driven by another satisfies `a & wave(b) == b & wave(a)` without invoking any spacetime metric. Because `.adjugate()` pulls back across incidence pairing (`b & wave.adjugate()(a) == a & wave(b)`), a symmetric map from potentials to currents equals its own adjugate. By contrast, `.adjoint()` pulls back across the metric scalar product (`wave.adjoint()(c) | a == c | wave(a)`), mapping currents back to potentials with opposite signs on timelike and spacelike axes.

In [ ]:
frequency = 1.0
probe = mv.t * frequency + mv.x * 0.9 + mv.y * 0.6 + mv.z * 0.4               # [] Vector+

In [ ]:
probe_wave = potential_map(media[-1], probe)                                  # [] Antivector- <- Vector+
probe_wave_adjoint = probe_wave.adjoint()                                     # [] Vector+ <- Antivector-
probe_wave_adjugate = probe_wave.adjugate()                                   # [] Antivector- <- Vector+
print(f"potential map\n{probe_wave.coefficient_table()}\n")
print(f"adjoint\n{probe_wave_adjoint.coefficient_table()}\n")
print(f"adjugate\n{probe_wave_adjugate.coefficient_table()}\n")

Matching entries across `probe_wave` and `probe_wave_adjugate` show that the potential map equals its incidence pullback entry for entry, whereas `probe_wave_adjoint` runs from currents back to potentials and flips signs along the timelike column and row.

That incidence symmetry comes from the factored structure `-probe_field.adjugate()(medium(probe_field))`. For the field map `probe_field = probe ^ Vector`, anticommutativity `a ^ probe == -(probe ^ a)` gives `probe_field.adjugate() == -(probe ^ Antibivector)`, while each constitutive dyad pairs two fields symmetrically through `a & (patch * (b & patch)) == b & (patch * (a & patch))`.

In the tensor index notation of premetric electrodynamics, the wave operator $M^{ab} = \chi^{acbd} k_c k_d$ is symmetric in $a$ and $b$ whenever $\chi^{acbd} = \chi^{bdac}$, whereas forming the metric adjoint requires lowering and raising indices with the spacetime metric $g_{ae}\, M^{ef}\, g_{fb}$.

In [ ]:
probe_field = probe ^ Vector                                                  # [] Bivector+ <- Vector+
probe_field_adjugate = probe_field.adjugate()                                 # [] Antivector- <- Antibivector-
print(f"field map\n{probe_field.coefficient_table()}\n")
print(f"adjugate\n{probe_field_adjugate.coefficient_table()}\n")

Taking `.adjugate()` on `probe_field` exchanges rows and columns while trading each basis blade for its spacetime complement, such as `xyz` for `t` and `yz` for `xt`, carrying the orientation signs of the regressive pairing.

In matrix notation on chosen spacetime bases, ordering the 6-dimensional excitation basis dual to the field basis identifies the pullback of $F(A) = k \wedge A$ with a signed transpose $F^{\mathsf{T}}$ so the wave operator factors as $-F^{\mathsf{T}} \chi F$.

## Wavevector sheets and meridian sections

Along any unit spatial wavefront at fixed frequency, a rest-frame reciprocal dielectric produces an even quartic in the wavenumber, reducing the dispersion relation to a quadratic in the squared wavenumber. Evaluating `fresnel_quartic` at three squared wavenumbers fixes that quadratic and recovers both positive polarization roots across a sphere of wavefronts and along its xz meridian circle.

In [ ]:
n_latitudes = 33
n_longitudes = 2 * (n_latitudes - 1) + 1
n_section_samples = 4 * (n_latitudes - 1)

In [ ]:
def sphere(n_latitudes: int, n_longitudes: int) -> Vector:
    """Polar and azimuthal rotor turns sweep a unit spatial plane across a sphere of wavefronts (`[n_latitudes, n_longitudes] Vector+`)."""
    polar = np.linspace(0, np.pi, n_latitudes)
    azimuth = np.linspace(0, 2 * np.pi, n_longitudes)
    meridian_rotors = (mv.zx * (-polar[:, None] / 2)).exp()                   # [n_latitudes, 1] Rotor+
    meridian = meridian_rotors >> mv.z                                        # [n_latitudes, 1] Vector+
    equatorial_rotors = (mv.xy * (-azimuth[None, :] / 2)).exp()               # [1, n_longitudes] Rotor+
    return equatorial_rotors >> meridian                                      # [n_latitudes, n_longitudes] Vector+


def circle(n_samples: int) -> Vector:
    """Rotor turns in the xz plane trace a circle of unit spatial wavefronts (`[n_samples] Vector+`)."""
    angles = np.linspace(0, 2 * np.pi, n_samples, endpoint=False)
    rotors = (mv.zx * (-angles / 2)).exp()                                    # [n_samples] Rotor+
    return rotors >> mv.z                                                     # [n_samples] Vector+

In [ ]:
def radial_sheets(medium: Constitutive, wavefronts: Vector, frequency: float) -> Vector:
    """Quadratic interpolation in the squared wavenumber yields both polarization sheets (`[..., 2] Vector+`) along unit spatial `wavefronts` (`[...] Vector+`)."""
    wavenumber_squared = np.arange(3)
    probes = mv.t * frequency + wavefronts[..., None] * np.sqrt(wavenumber_squared)  # [..., 3] Vector+
    values = fresnel_quartic(medium[..., None], time, rest_space, probes)     # [..., 3] Scalar+
    constant, unit, twice = (values[..., i] for i in range(3))                # [...] Scalar+, Scalar+, Scalar+
    quadratic = (twice - 2 * unit + constant) / 2                             # [...] Scalar+
    linear = unit - constant - quadratic                                      # [...] Scalar+
    # Where the sheets coincide the discriminant vanishes up to round-off of either sign.
    discriminant = (linear.squared() - 4 * quadratic * constant).abs().square_root()  # [...] Scalar+
    wavenumbers_squared = (-linear[..., None] + discriminant[..., None] * root_signs) / (2 * quadratic[..., None])  # [..., 2] Scalar+
    return wavefronts[..., None] * wavenumbers_squared.square_root()          # [..., 2] Vector+


wavefronts = sphere(n_latitudes, n_longitudes)                                # [n_latitudes, n_longitudes] Vector+
section_wavefronts = circle(n_section_samples)                                # [n_section_samples] Vector+
sheets = radial_sheets(media[:, None, None], wavefronts, frequency)           # [n_materials, n_latitudes, n_longitudes, 2] Vector+
sections = radial_sheets(media[:, None], section_wavefronts, frequency)       # [n_materials, n_section_samples, 2] Vector+
save_figure(render.comparison_figure(sheets, sections), "fresnel");

From left to right, the three columns display vacuum, the uniaxial crystal, and the biaxial crystal, with the inner polarization sheet in solid blue and the outer sheet in orange (dashed in the lower xz meridian sections). Both sheets coincide as a single sphere in vacuum, touch along the vertical optic axis in the uniaxial crystal, and intersect at four conical degeneracy points along two optic axes in the xz plane of the biaxial crystal.

## The axion coupling

An axion medium adds an excitation term `coupling * Bivector` directly proportional to the field bivector itself. Because fields are extrinsic bivectors while excitations are intrinsic antibivectors, the proportionality constant `coupling` carries intrinsic scalar orientation (`Scalar-`) and reverses sign under mirror reflection; multiplying `Bivector` by an ordinary extrinsic scalar produces an extrinsic map whose sum with `media` is rejected by the type system.

In a homogeneous medium, wedging the axion excitation with the wavevector yields `k ^ (coupling * (k ^ A)) == 0` identically because `k ^ k == 0`. Bulk dispersion therefore remains unchanged across all three media, and the axion response appears physically only across interfaces or spatial gradients where `coupling` varies.

In [ ]:
axion_coupling = 0.3

In [ ]:
coupling = mv(AxionCoupling, [axion_coupling])                                # [] Scalar-
axion = coupling * Bivector                                                   # [] Antibivector- <- Bivector+
coupled_sheets = radial_sheets((media + axion)[:, None, None], wavefronts, frequency)  # [n_materials, n_latitudes, n_longitudes, 2] Vector+
sheet_wavevectors = mv.t * frequency + sheets                                 # [n_materials, n_latitudes, n_longitudes, 2] Vector+
sheet_waves = potential_map(media[:, None, None, None], sheet_wavevectors)    # [n_materials, n_latitudes, n_longitudes, 2] Antivector- <- Vector+
axion_waves = potential_map(axion, sheet_wavevectors)                         # [n_materials, n_latitudes, n_longitudes, 2] Antivector- <- Vector+
reflected_axion = mv.x >> axion(mv.x << Bivector)                             # [] Antibivector- <- Bivector+

In [ ]:
# checks
# Incidence symmetry and factored pullback of the probe potential map.
assert (np.abs((probe_wave_adjugate - probe_wave).kernel) < 1e-12).all()
factored_probe_wave = -probe_field_adjugate(media[-1](probe_field))           # [] Antivector- <- Vector+
assert (np.abs((factored_probe_wave - probe_wave).kernel) < 1e-12).all()
# Quartic roots, gauge null space, and symmetry across all three wave surfaces.
quartic_residuals = fresnel_quartic(media[:, None, None, None], time, rest_space, sheet_wavevectors)  # [n_materials, n_latitudes, n_longitudes, 2] Scalar+
assert (quartic_residuals.abs() < 1e-9).all()
assert (np.abs(sheet_waves(sheet_wavevectors).kernel) < 1e-11).all()
assert (np.abs((sheet_waves.adjugate() - sheet_waves).kernel) < 1e-12).all()
# Bulk invisibility and mirror-odd parity of the chiral axion coupling.
assert (np.abs(axion_waves.kernel) < 1e-12).all()
assert (np.abs((coupled_sheets - sheets).kernel) < 1e-6).all()
assert (np.abs((reflected_axion + axion).kernel) < 1e-12).all()